In [2]:
import numpy as np
import pandas as pd
import os

In [3]:
root_dir="C:\\Users\\Hp\\Downloads\\archive"
paths=[]
emotions=[]
emotion_dict = {
    '01':0,
    '02':1,
    '03':2,
    '04':3,
    '05':4,
    '06':5,
    '07':6,
    '08':7
}
for root,dir,files in os.walk(root_dir):
    for file in files:
        if file.endswith(".wav"):
            emotion=emotion_dict[file.split("-")[2]]
            path=os.path.join(root,file)
            emotions.append(emotion)
            paths.append(path)
df=pd.DataFrame()
df["path"]=paths
df["emotion"]=emotions
print(df.head())


                                                path  emotion
0  C:\Users\Hp\Downloads\archive\Actor_01\03-01-0...        0
1  C:\Users\Hp\Downloads\archive\Actor_01\03-01-0...        0
2  C:\Users\Hp\Downloads\archive\Actor_01\03-01-0...        0
3  C:\Users\Hp\Downloads\archive\Actor_01\03-01-0...        0
4  C:\Users\Hp\Downloads\archive\Actor_01\03-01-0...        1


In [4]:
print(df["emotion"].value_counts())

emotion
1    384
2    384
3    384
4    384
6    384
5    384
7    384
0    192
Name: count, dtype: int64


In [5]:
## Predicting emotion using CNN
import librosa
import torch 
import torch.nn as nn

In [13]:


def extract_features(file_path):
    target_duration = 3
    sr = 22050
    signal, _ = librosa.load(file_path, sr=sr, duration=target_duration)
    if len(signal) < sr * target_duration:
        signal = librosa.util.fix_length(signal, size=sr * target_duration)
    features = librosa.feature.melspectrogram(y=signal, sr=sr, n_mels=128)
    features_db = librosa.power_to_db(features, ref=np.max)
    
    return features_db

In [14]:
X = []
y = []
for path, emotion in zip(df.path, df.emotion):
    features = extract_features(path)
    X.append(features)
    y.append(emotion)

In [15]:
from sklearn.model_selection import train_test_split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42
)
X_val,X_test,y_val,y_test=train_test_split(
    X_temp,y_temp,test_size=0.5,random_state=42
)
X_train = torch.tensor(X_train).float()
X_test = torch.tensor(X_test).float()
y_train = torch.tensor(y_train)
y_test = torch.tensor(y_test)
y_val=torch.tensor(y_val)
X_val=torch.tensor(X_val).float()

C:\Users\Hp\AppData\Local\Temp\ipykernel_13200\1883104782.py:8: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  X_train = torch.tensor(X_train).float()


In [16]:
class CNN(nn.Module):
    def __init__(self,):
        super().__init__()
        self.conv1=nn.Conv2d(1,16,3,1,1)
        self.conv2=nn.Conv2d(16,32,3,1,1)
        self.conv3=nn.Conv2d(32,32,3,1,1)
        self.relu=nn.ReLU()
        self.pool=nn.MaxPool2d(2,2)
        self.fc1=nn.Linear(32*16*16,8)
    def forward(self,x):
        x=self.conv1(x)
        x=self.relu(x)
        x=self.pool(x)

        x=self.conv2(x)
        x=self.relu(x)
        x=self.pool(x)

        x=self.conv3(x)
        x=self.relu(x)
        x=self.pool(x)

        x=x.view(x.size(0),-1)
        predict=self.fc1(x)
        return predict

In [17]:
model=CNN()
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=0.0001)

In [199]:
train_loss_CNN=[]
val_loss_CNN=[]
model.train()
for epoch in range(5):
    total_loss=0.0
    val_loss=0.0
    for i in range(len(X_train)):
        inputs = X_train[i].unsqueeze(0).unsqueeze(0)
        labels = y_train[i].unsqueeze(0)

        optimizer.zero_grad()
        outputs=model(inputs)
        loss=criterion(outputs,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss.item()
    for i in range(len(X_val)):
        inputs1 = X_val[i].unsqueeze(0).unsqueeze(0)
        labels1 = y_val[i].unsqueeze(0)
        outputs1=model(inputs1)
        loss1=criterion(outputs1,labels)
        val_loss+=loss1.item()
    train_loss_CNN.append(total_loss/len(X_train))
    val_loss_CNN.append(val_loss/len(X_val))
    print(f"Epoch {epoch+1} loss : {total_loss}")
# print(y_train.shape)

Epoch 1 loss : 80.48145122017138
Epoch 2 loss : 79.06499469906589
Epoch 3 loss : 61.3801488949507
Epoch 4 loss : 39.280655351624084
Epoch 5 loss : 1.2970743696763307


In [200]:
model.eval()
correct=0
with torch.no_grad():
    for i in range(len(X_test)):
        inputs= X_test[i].unsqueeze(0).unsqueeze(0) 
        labels = y_test[i].unsqueeze(0)
        outputs=model(inputs)
        prediction=torch.argmax(outputs)
        correct+=(prediction==labels)
# print(outputs[0][1])

In [201]:
print(f"Accuracy : {correct*100/len(y_test)}")
print(correct,len(y_test))

Accuracy : tensor([87.5000])
tensor([378]) 432


In [202]:
print(train_loss_CNN)
print(val_loss_CNN)

[0.03992135477191041, 0.0392187473705684, 0.030446502428050942, 0.019484452059337343, 0.0006433900643235767]
[8.397381775239266, 14.523674122299727, 16.037222489773413, 14.92299088337958, 15.284840282182445]


In [20]:
## Predicting Emotion using LSTM
import whisper
model=whisper.load_model("base.en")
def extract_transcript(model,dataset):
    Xs=[]
    y=[]
    for idx,rows in dataset.iterrows():
        result=model.transcribe(rows["path"])
        Xs.append(result["text"])
        y.append(rows["emotion"])
    return Xs,y

In [21]:
Xk,y_RNN=extract_transcript(model,df)
print(Xk)

c:\Users\Hp\AppData\Local\Programs\Python\Python314\Lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


[' Kids are talking by the door.', ' Kids are talking by the door.', ' Dogs are sitting by the door.', ' Dogs are sitting by the door.', ' Kids are talking by the door.', ' Kids are talking by the door.', ' Dogs are sitting by the door.', ' Dogs are sitting by the door.', ' Kids are talking by the door.', ' Kids are talking by the door.', ' Dogs are sitting by the door.', ' Dogs are sitting by the door.', ' Kids are talking by the door.', ' Kids are talking by the door.', ' Dogs are sitting by the door.', ' Dogs are sitting by the door.', ' Kids are talking by the door.', ' Kids are talking by the door.', ' Dogs are sitting by the door.', ' Dogs are sitting by the door.', ' Kids are talking by the door.', ' Kids are talking by the door.', ' Dogs are sitting by the door.', ' Dogs are sitting by the door.', ' Kids are talking by the door.', ' Kids are talking by the door.', ' Dogs are sitting by the door.', ' Dogs are sitting by the door.', ' Kids are talking by the door.', ' Kids are ta

In [22]:
data = {
    'transcript': Xk,
    'emotion': y_RNN
}
df_res=pd.DataFrame(data)
df_res.to_csv("whisper_transcripts.csv", index=False)

In [18]:
df_RNN=pd.read_csv('C:\\Users\\Hp\\OneDrive\\Desktop\\python_learning\\whisper_transcripts.csv')
X_RNN=df_RNN["transcript"].values
y_RNN=df_RNN["emotion"].values

In [19]:
## Tokenization and padding
li=list(set(X_RNN))
all_sen=''.join(li)
import re
preprocessed=re.split(r'([,.:;?_!"()\']|--|\s)',all_sen)
preprocessed=[item.strip() for item in preprocessed if item.strip()]
all_words=sorted(set(preprocessed))
vocab={token: inte for inte,token in enumerate(all_words)}


In [20]:
def text_to_seq(x,vocab=vocab,max_length=20):
    Xrr=[]
    for i in x:
        words=re.split(r'([,.:;?_!"()\']|--|\s)',i)
        words=[item.strip() for item in words if item.strip()]
        seq=[]
        for k in words:
            seq.append(vocab[k])
        if len(seq) < max_length:
            seq += [0] * (max_length - len(seq))
        else:
            seq = seq[:max_length]
        Xrr.append(seq)
    return Xrr
    

In [21]:
X_ls=text_to_seq(X_RNN)

X_train_ls, X_temp_ls, y_train_ls, y_temp_ls = train_test_split(
    X_ls, y_RNN, test_size=0.3, random_state=42
)
X_test_ls,X_val_ls,y_test_ls,y_val_ls=train_test_split(
    X_temp_ls,y_temp_ls,test_size=0.5,random_state=42
)
X_train_ls = torch.tensor(X_train_ls).long()
X_test_ls = torch.tensor(X_test_ls).long()
y_train_ls = torch.tensor(y_train_ls)
y_test_ls = torch.tensor(y_test_ls)
X_val_ls=torch.tensor(X_val_ls).long()
y_val_ls=torch.tensor(y_val_ls)

In [22]:
class LSTM(nn.Module):
    def __init__(self,size,emb_dim,hidden_dim,output_dim):
        super().__init__()
        self.embedding=nn.Embedding(size,emb_dim)
        self.lstm=nn.LSTM(emb_dim,hidden_dim,batch_first=True)
        self.fc=nn.Linear(hidden_dim,output_dim)
    def forward(self,x):
        embedded=self.embedding(x)
        _,(hidden,_)=self.lstm(embedded)
        out=self.fc(hidden[-1])
        return out        


In [23]:
model_ls=LSTM(73,16,64,8)
criterion_ls=nn.CrossEntropyLoss()
optimizer_ls=torch.optim.Adam(model_ls.parameters(),lr=0.001)

In [24]:
train_loss_LS=[]
val_loss_LS=[]
model_ls.train()
for epoch in range(5):
    epoch_loss=0.0
    val_ls_loss=0.0
    for i in range(len(X_train_ls)):
        optimizer_ls.zero_grad()
        outputs_ls=model_ls(X_train_ls[i].unsqueeze(0))
        loss_ls=criterion_ls(outputs_ls,y_train_ls[i].unsqueeze(0))
        loss_ls.backward()
        optimizer_ls.step()
        epoch_loss+=loss_ls.item()
    for i in range(len(X_val_ls)):
        outputs1_ls=model_ls(X_val_ls[i].unsqueeze(0))
        loss1_ls=criterion_ls(outputs1_ls,y_val_ls[i].unsqueeze(0))
        val_ls_loss+=loss1_ls.item()
    train_loss_LS.append(epoch_loss/len(X_train_ls))
    val_loss_LS.append(val_ls_loss/len(X_val_ls))
    print(f"Epoch {epoch+1} loss = {epoch_loss}")

Epoch 1 loss = 4169.827446818352
Epoch 2 loss = 4158.834023475647
Epoch 3 loss = 4156.684489250183
Epoch 4 loss = 4155.505447983742
Epoch 5 loss = 4154.720922231674


In [26]:
model_ls.eval()
correct_ls=0
with torch.no_grad():
    for i in range(len(X_test_ls)):
        out=model_ls(X_test_ls[i].unsqueeze(0))
        pre=torch.argmax(out)
        correct_ls+=(pre==y_test_ls[i])

        

In [27]:
print(f"Accuracy : {correct_ls*100/len(y_test_ls)}")
print(correct_ls,len(y_test_ls))

Accuracy : 12.5
tensor(54) 432


In [28]:
print(train_loss_LS)
print(val_loss_LS)

[2.068366789096405, 2.0629137021208566, 2.0618474649058447, 2.0612626230078086, 2.0608734733292033]
[2.0679642718147346, 2.066108945619177, 2.065437396367391, 2.06513859745529, 2.0649956034289465]


In [ ]:
## Fusion model
## X_ls,y_RNN - LSTM model
## X,y -CNN model
import random
random.seed(42)
test_idx= random.sample(range(2880),576)
indices=[i for i in range(2880)]
train_idx=sorted(list(set(indices) - set(test_idx)))

X_train_fuls=[]
X_test_fuls=[]
X_train_fucn=[]
X_test_fucn=[]
y_train_fu=[]
y_test_fu=[]
for i in train_idx:
    X_train_fuls.append(X_ls[i])
    X_train_fucn.append(X[i])
    y_train_fu.append(y[i])
for i in test_idx:
    X_test_fuls.append(X_ls[i])
    X_test_fucn.append(X[i])
    y_test_fu.append(y[i])

X_train_fuls=torch.tensor(X_train_fuls).long()
X_test_fuls=torch.tensor(X_test_fuls).long()

X_train_fucn=torch.tensor(X_train_fucn).float()
X_test_fucn=torch.tensor(X_test_fucn).float()

y_train_fu=torch.tensor(y_train_fu)
y_test_fu=torch.tensor(y_test_fu)

In [159]:
## LSTM train
model_ls.train()
for epoch in range(5):
    epoch_loss=0.0
    for i in range(len(X_train_fuls)):
        optimizer_ls.zero_grad()
        outputs_ls=model_ls(X_train_fuls[i].unsqueeze(0))
        loss_ls=criterion_ls(outputs_ls,y_train_fu[i].unsqueeze(0))
        loss_ls.backward()
        optimizer_ls.step()
        epoch_loss+=loss_ls.item()
    print(f"Epoch {epoch+1} loss LSTM model = {epoch_loss}")



Epoch 1 loss LSTM model = 4753.567086100578
Epoch 2 loss LSTM model = 4753.173688530922
Epoch 3 loss LSTM model = 4753.1064656972885
Epoch 4 loss LSTM model = 4753.093243360519
Epoch 5 loss LSTM model = 4753.089976429939


In [167]:
cnn_model=CNN()
optimizer=torch.optim.Adam(cnn_model.parameters(),lr=0.0001)
cnn_model.train()
for epoch in range(10):
    total_loss=0.0
    for i in range(len(X_train_fucn)):
        inputs = X_train_fucn[i].unsqueeze(0).unsqueeze(0)
        labels = y_train_fu[i].unsqueeze(0)

        optimizer.zero_grad()
        outputs=cnn_model(inputs)
        loss=criterion(outputs,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss.item()
    print(f"Epoch {epoch+1} loss CNN model : {total_loss}")

Epoch 1 loss CNN model : 5252.006558536086
Epoch 2 loss CNN model : 4707.827062252909
Epoch 3 loss CNN model : 4098.506251748651
Epoch 4 loss CNN model : 3502.5067446161993
Epoch 5 loss CNN model : 3050.825387987308
Epoch 6 loss CNN model : 2691.429769269773
Epoch 7 loss CNN model : 2386.7901575329415
Epoch 8 loss CNN model : 2016.1132585629966
Epoch 9 loss CNN model : 1627.420047213067
Epoch 10 loss CNN model : 1266.6776295909926


In [170]:
cnn_model.eval()
model_ls.eval()
with torch.no_grad():
    correct=0
    for i in range(len(X_test_fucn)):
        inputs= X_test_fucn[i].unsqueeze(0).unsqueeze(0) 
        labels = y_test_fu[i].unsqueeze(0)
        outputs_cnn=cnn_model(inputs)
        outputs_ls=model_ls(X_test_fuls[i].unsqueeze(0))
        prediction=0.9*outputs_cnn+0.1*outputs_ls
        prediction=torch.argmax(prediction)
        correct+=(prediction==y_test_fu[i])

In [171]:
print(correct*100/len(y_test_fu))

tensor(73.9583)
